**U24AI040 NLP LAB 6**

Use the 4 models you developed in the previous assignment and test it on your test and
validation/development sets. Design the following smoothing techniques.
1. Interpolated Smoothing (For Bigram, Trigram, and Quadgram Models)
2. Good Turing Smoothing
3. Katz Backoff Smoothing
4. Stupid Backoff Smoothing
5. Kneser-Ney Smoothing

In [221]:
import math
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

In [222]:
DATA_PATH = Path(
    r"C:/Users/VIVEK/Desktop/NLP Lab/ASSIGNMENT_1/Q1/tokenized_gujarati_corpus/10lakh sentence.parquet"
)

df = pd.read_parquet(DATA_PATH, columns=["tokens"])

print("Total sentences:", len(df))

Total sentences: 1099721


In [223]:
print(df.head())
print(df["tokens"].iloc[0])
print("Tokens in first sentence:", len(df["tokens"].iloc[0]))

                                              tokens
0  [આ, વીડિયો, જુઓ, :, ઊંઝા, માર્કેટયાર્ડ, આજથી, ...
1                       [મિથેનોલ, આવ્યો, ક્યાંથી, ?]
2  [આખરે, ત્રણ, રાજ્યોમાં, મળેલ, હાર, પર, કોંગ્રે...
3  [તેમણે, કહ્યું, કે, ,, ત્રિપુરા, ,, નાગાલેન્ડ,...
4  [આ, આંકડો, માટે, ,, અને, વજન, ઘટાડવા, માટે, પ્...
['આ' 'વીડિયો' 'જુઓ' ':' 'ઊંઝા' 'માર્કેટયાર્ડ' 'આજથી' '25' 'જુલાઈ' 'સુધી'
 'બંધ']
Tokens in first sentence: 11


In [224]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

dev_df = df.iloc[:1000].copy()
test_df = df.iloc[1000:2000].copy()
train_df = df.iloc[2000:].copy()

print("Total sentences:", len(df))
print("Training sentences:", len(train_df))
print("Development sentences:", len(dev_df))
print("Test sentences:", len(test_df))

if len(df) < 1_000_000:
    print("WARNING: Fewer than 1,000,000 sentences are available.")
else:
    print("Requirement satisfied: at least 1,000,000 sentences.")


Total sentences: 1099721
Training sentences: 1097721
Development sentences: 1000
Test sentences: 1000
Requirement satisfied: at least 1,000,000 sentences.


In [225]:
def most_common(counter):
    
    if not counter:
        return None
    
    return counter.most_common(1)[0][0]

In [226]:
def train_model(sentences):
  unigram=Counter()
  bigram=defaultdict(Counter)
  trigram=defaultdict(Counter)
  quadgram=defaultdict(Counter)

  for tokens in sentences:
    tokens=list(tokens)

    for word in tokens:
      unigram[word]+=1

    unigram["</s>"]+=1

    tokens2=["<s>"] + tokens + ["</s>"]

    for i in range(len(tokens2)-1):
        w1=tokens2[i]
        w2=tokens2[i+1]
        bigram[w1][w2]+=1

    tokens3=["<s>","<s>"]+tokens+["</s>"]
    for i in range(len(tokens3)-2):
       w=tokens3[i]
       w1=tokens3[i+1]
       w2=tokens3[i+2]
       trigram[(w,w1)][w2]+=1

    tokens4=["<s>","<s>","<s>"]+tokens+["</s>"]
    for i in range(len(tokens4)-3):
        w1 = tokens4[i]
        w2 = tokens4[i + 1]
        w3 = tokens4[i + 2]
        w4 = tokens4[i + 3]

        quadgram[(w1, w2, w3)][w4] += 1

  return unigram,bigram,trigram,quadgram



In [227]:
unigram,bigram,trigram,quadgram=train_model(train_df["tokens"])

V = len(unigram) - 1

print("Prediction vocabulary size:", V)
print("Number of unigram entries:", len(unigram))
print("Number of bigram contexts:", len(bigram))
print("Number of trigram contexts:", len(trigram))
print("Number of quadrigram contexts:", len(quadgram))

Prediction vocabulary size: 514766
Number of unigram entries: 514767
Number of bigram contexts: 514767
Number of trigram contexts: 5453509
Number of quadrigram contexts: 10690894


**UnSmoothed**

In [228]:
def unigram_prob(word,unigram):
  total=sum(unigram.values())
  return unigram[word]/total

In [229]:
def bigram_prob(w1,w2,bigram):
  total=sum(bigram[w1].values())
  if total == 0:
      return 0
  c=bigram[w1][w2]
  return c/total


In [230]:
def trigram_prob(w1,w2,w3,trigram):
  total=sum(trigram[(w1,w2)].values())
  if total == 0:
        return 0
  c=trigram[(w1,w2)][w3]
  return c/total

In [231]:
def quadgram_prob(w1, w2, w3, w4, quadgram):
    total = sum(quadgram[(w1, w2, w3)].values())
    if total == 0:
        return 0
    c = quadgram[(w1, w2, w3)][w4]

    return c / total

In [232]:
def predict_bigram(word,bigram):
  word=bigram[word]
  return most_common(word)



In [233]:
def predict_trigram(w1,w2,trigram):
  word=trigram[(w1,w2)]
  return most_common(word)


In [234]:
def predict_quadgram(w1,w2,w3,quadgram):
  word=quadgram[(w1,w2,w3)]
  return most_common(word)

In [ ]:
context = "ગુજરાતી"

prediction = predict_bigram(context,bigram)

print("Previous word:",context)

print("Predicted next word:",prediction)

Previous word: ગુજરાતી
Predicted next word: ફિલ્મ


In [ ]:
word1 = "મને"
word2 = "ગુજરાતી"

prediction = predict_trigram(word1,word2,trigram)

print("Previous words:",word1,word2)

print("Predicted next word:",prediction)

Previous words: મને ગુજરાતી
Predicted next word: વાનગીઓ


In [ ]:
word1 = "મને"
word2 = "ગુજરાતી"
word3 = "વાનગીઓ"

prediction = predict_quadgram(word1,word2,word3,quadgram)

print("Previous words:",  word1,word2,word3)

print("Predicted next word:",prediction)

Previous words: મને ગુજરાતી વાનગીઓ
Predicted next word: ખવડાવવાનું


In [238]:
def accuracy(sentences,order,bigram,trigram,quadgram):
  correct=0
  total=0
  for tokens in sentences:
    tokens=list(tokens)

    if order==2:
      for i in range(len(tokens)-1):
        w1=tokens[i]
        w2=tokens[i+1]
        pred=predict_bigram(w1,bigram)
        if pred==w2:
          correct+=1
        total+=1

    if order==3:
      for i in range(len(tokens)-2):
          w1=tokens[i]
          w2=tokens[i+1]
          w3=tokens[i+2]
          pred=predict_trigram(w1,w2,trigram)
          if pred==w3:
            correct+=1
          total+=1

    if order==4:
      for i in range(len(tokens)-3):
          w1=tokens[i]
          w2=tokens[i+1]
          w3=tokens[i+2]
          w4=tokens[i+3]
          pred=predict_quadgram(w1,w2,w3,quadgram)
          if pred==w4:
            correct+=1
          total+=1

  if total==0:
    return 0

  return correct/total


In [239]:
bigram_test_accuracy = accuracy(
    test_df["tokens"],2,bigram,trigram,quadgram)

In [240]:
def get_ngrams(tokens,n):
  tokens= (n-1)*["<s>"] + tokens + ["</s>"]
  ngram=[]
  for i in range(len(tokens) - (n-1)):
    curr=[]

    for j in range(n):
      curr.append(tokens[i+j])

    ngram.append(curr)

  return ngram

In [ ]:
def perplexity(sentence,order,unigram,bigram,trigram,quadgram):
  log_prob=0
  total_word=0
  for tokens in sentence:
    tokens=list(tokens)
    
    tokens= (order-1)*["<s>"] + tokens + ["</s>"]
    for i in range(len(tokens)-(order-1)):
      prob=0
      if order==1:
        prob=unigram_prob(tokens[i],unigram)
      elif order==2:
        prob=bigram_prob(tokens[i],tokens[i+1],bigram)
      elif order==3:
        prob=trigram_prob(tokens[i],tokens[i+1],tokens[i+2],trigram)
      elif order==4:
        prob=quadgram_prob(tokens[i],tokens[i+1],tokens[i+2],tokens[i+3],quadgram)

      if prob==0:
        return float('inf')

      log_prob+=math.log(prob)
      total_word+=1

  return math.exp(-log_prob/total_word)

In [ ]:
unigram_test_pp = perplexity(test_df["tokens"],1,unigram,bigram,trigram,quadgram)
bigram_test_pp = perplexity(test_df["tokens"],2,unigram,bigram,trigram,quadgram)
print(unigram_test_pp)
print(bigram_test_pp)

inf
inf


## Interpolated Smoothing

In [243]:
l1=0.1
l2=0.2
l3=0.3
l4=0.4


In [244]:
def interpolated_quad_prob(w1,w2,w3,w4,quadgram,trigram,bigram,unigram,l1,l2,l3,l4):
  return l4*quadgram_prob(w1,w2,w3,w4,quadgram)+l3*trigram_prob(w2,w3,w4,trigram)+l2*bigram_prob(w3,w4,bigram)+l1*unigram_prob(w4,unigram)

def interpolated_tri_prob(w1,w2,w3,trigram,bigram,unigram,l1,l2,l3):
  return l3*trigram_prob(w1,w2,w3,trigram)+l2*bigram_prob(w2,w3,bigram)+l1*unigram_prob(w3,unigram)

def interpolated_bi_prob(w1,w2,bigram,unigram,l1,l2):
  return l2*bigram_prob(w1,w2,bigram)+l1*unigram_prob(w2,unigram)

def interpolated_uni_prob(w1,unigram,l1):
  return unigram_prob(w1,unigram)


**perplexity**

In [245]:
def interpolated_perplexity(sentence,order,unigram,bigram,trigram,quadgram,l1=0.1,l2=0.2,l3=0.3,l4=0.4):
  log_prob=0
  total_word=0
  for tokens in sentence:
    tokens=list(tokens)
    
    tokens= (order-1)*["<s>"] + tokens + ["</s>"]
    for i in range(len(tokens)-(order-1)):
      prob=0
      if order==1:
        prob=interpolated_uni_prob(tokens[i],unigram,1)
      elif order==2:
        prob=interpolated_bi_prob(tokens[i],tokens[i+1],bigram,unigram,l1,l2)
      elif order==3:
        prob=interpolated_tri_prob(tokens[i],tokens[i+1],tokens[i+2],trigram,bigram,unigram,l1,l2,l3)
      elif order==4:
        prob=interpolated_quad_prob(tokens[i],tokens[i+1],tokens[i+2],tokens[i+3],quadgram,trigram,bigram,unigram,l1,l2,l3,l4)

      if prob<=0:
        return float('inf')

      log_prob+=math.log(prob)
      total_word+=1

  return math.exp(-log_prob/total_word)


      
  

In [ ]:
bigram_dev_pp = interpolated_perplexity(dev_df["tokens"],2,unigram,bigram,trigram,quadgram,l1=0.2,l2=0.8)

bigram_test_pp = interpolated_perplexity(test_df["tokens"],2,unigram,bigram,trigram,quadgram,l1=0.2,l2=0.8)

print("Bigram Dev PP:", bigram_dev_pp)
print("Bigram Test PP:", bigram_test_pp)

Bigram Dev PP: inf
Bigram Test PP: inf


In [ ]:
train_vocab = set(unigram.keys())
unseen = 0
total = 0

for tokens in test_df["tokens"]:
    for word in tokens:
        total += 1
        if word not in train_vocab:
            unseen += 1

print("Total test words:", total)
print("Unseen test words:", unseen)
print("Unseen percentage:", unseen / total * 100)

Total test words: 14720
Unseen test words: 278
Unseen percentage: 1.888586956521739


**Accuracy**

In [ ]:
def interpolated_accuracy(sentences,order,unigram,bigram,trigram,quadgram,l1,l2,l3=0,l4=0):
    correct = 0
    total = 0

    unigram_total = sum(unigram.values())

    for tokens in sentences:

        tokens = list(tokens)
        tokens = (order - 1) * ["<s>"] + tokens

        for i in range(order - 1, len(tokens)):

            actual = tokens[i]

            if order == 2:

                w1 = tokens[i - 1]
                candidates = bigram[w1]

                if not candidates:
                    continue

                context_total = sum(candidates.values())

                best_word = None
                best_prob = -1

                for word, count in candidates.items():

                    p_bigram = count / context_total
                    p_unigram = unigram.get(word, 0) / unigram_total

                    prob = (l2 * p_bigram+l1 * p_unigram)

                    if prob > best_prob:
                        best_prob = prob
                        best_word = word

            elif order == 3:

                w1 = tokens[i - 2]
                w2 = tokens[i - 1]

                candidates = trigram[(w1, w2)]

                if not candidates:
                    continue

                trigram_total = sum(candidates.values())

                bigram_candidates = bigram[w2]
                bigram_total = sum(bigram_candidates.values())

                best_word = None
                best_prob = -1

                for word, count in candidates.items():

                    p_tri = count / trigram_total

                    p_bi = (bigram_candidates.get(word, 0)/ bigram_total if bigram_total > 0 else 0)

                    p_uni = (unigram.get(word, 0)/ unigram_total)

                    prob = (l3 * p_tri+ l2 * p_bi+ l1 * p_uni)

                    if prob > best_prob:
                        best_prob = prob
                        best_word = word
            elif order == 4:

                w1 = tokens[i - 3]
                w2 = tokens[i - 2]
                w3 = tokens[i - 1]

                candidates = quadgram[(w1, w2, w3)]

                if not candidates:
                    continue

                quad_total = sum(candidates.values())

                tri_candidates = trigram[(w2, w3)]
                tri_total = sum(tri_candidates.values())

                bi_candidates = bigram[w3]
                bi_total = sum(bi_candidates.values())

                best_word = None
                best_prob = -1

                for word, count in candidates.items():

                    p_quad = count / quad_total

                    p_tri = (tri_candidates.get(word, 0) / tri_total if tri_total > 0
                        else 0
                    )

                    p_bi = (bi_candidates.get(word, 0) / bi_total if bi_total > 0
                        else 0
                    )

                    p_uni = (unigram.get(word, 0)/ unigram_total)

                    prob = (l4 * p_quad+ l3 * p_tri+ l2 * p_bi+ l1 * p_uni)

                    if prob > best_prob:
                        best_prob = prob
                        best_word = word

            else:
                raise ValueError("Order must be 2, 3, or 4")

            if best_word == actual:
                correct += 1

            total += 1

    return correct / total if total > 0 else 0

In [ ]:
bigram_acc = interpolated_accuracy(dev_df["tokens"],2,unigram,bigram,trigram,quadgram,l1=0.2,l2=0.8)

print("Bigram Dev Accuracy:", bigram_acc)

Bigram Dev Accuracy: 0.17477799958697599


In [ ]:
trigram_acc = interpolated_accuracy(dev_df["tokens"],3,unigram,bigram,trigram,quadgram,l1=0.1,l2=0.3,l3=0.6)

print("Trigram Dev Accuracy:", trigram_acc)

Trigram Dev Accuracy: 0.238003663003663


In [ ]:
quadgram_acc = interpolated_accuracy(dev_df["tokens"],4,unigram,bigram,trigram,quadgram,l1=0.05,l2=0.15,l3=0.30,l4=0.50)

print("Quadgram Dev Accuracy:", quadgram_acc)

Quadgram Dev Accuracy: 0.2710053423992229


## Good Turing

In [252]:
def create_bins(counts):
  Nc=Counter()

  for i in counts:
    Nc[i]+=1
  return Nc

In [253]:
uni_counts=[]

for word,count in unigram.items():
    uni_counts.append(count)

bi_count=[]

for context in bigram:

    for word,count in bigram[context].items():
        bi_count.append(count)


tri_count=[]

for context in trigram:

    for word,count in trigram[context].items():
        tri_count.append(count)


quad_count=[]

for context in quadgram:

    for word,count in quadgram[context].items():
        quad_count.append(count)

In [254]:
uni_Nc=create_bins(uni_counts)

bi_Nc=create_bins(bi_count)

tri_Nc=create_bins(tri_count)

quad_Nc=create_bins(quad_count)

In [255]:
print("Unigram:")
print("N1 =",uni_Nc[1])
print("N2 =",uni_Nc[2])
print("N3 =",uni_Nc[3])

print("\nBigram:")
print("N1 =",bi_Nc[1])
print("N2 =",bi_Nc[2])
print("N3 =",bi_Nc[3])

print("\nTrigram:")
print("N1 =",tri_Nc[1])
print("N2 =",tri_Nc[2])
print("N3 =",tri_Nc[3])

print("\nQuadgram:")
print("N1 =",quad_Nc[1])
print("N2 =",quad_Nc[2])
print("N3 =",quad_Nc[3])

Unigram:
N1 = 288666
N2 = 66584
N3 = 31557

Bigram:
N1 = 4237807
N2 = 562194
N3 = 210279

Trigram:
N1 = 9708972
N2 = 610569
N3 = 179812

Quadgram:
N1 = 12627776
N2 = 432838
N3 = 111273


In [314]:
def adjusted_counts(frequencies):

    adjusted = {}

    for count in sorted(frequencies):

        Nc = frequencies[count]
        Nc1 = frequencies.get(count + 1, 0)

        if Nc1 == 0:
            adjusted[count] = count
        else:
            adjusted[count] = (count + 1) * Nc1 / Nc

    return adjusted

In [315]:
uni_adjusted=adjusted_counts(uni_Nc)

bi_adjusted=adjusted_counts(bi_Nc)

tri_adjusted=adjusted_counts(tri_Nc)

quad_adjusted=adjusted_counts(quad_Nc)

In [316]:
print("Bigram adjusted counts:")

for c in [1,2,3]:

  if c in bi_adjusted:
    print("c =",c,"c* =",bi_adjusted[c])


print("\nTrigram adjusted counts:")

for c in [1,2,3]:

  if c in tri_adjusted:
    print("c =",c,"c* =",tri_adjusted[c])


print("\nQuadgram adjusted counts:")

for c in [1,2,3]:

  if c in quad_adjusted:
    print("c =",c,"c* =",quad_adjusted[c])

Bigram adjusted counts:
c = 1 c* = 0.2653230786583721
c = 2 c* = 1.1220984215413186
c = 3 c* = 2.1131924728574893

Trigram adjusted counts:
c = 1 c* = 0.12577418082985511
c = 2 c* = 0.8834971968770114
c = 3 c* = 1.8799635174515605

Quadgram adjusted counts:
c = 1 c* = 0.06855332245361337
c = 2 c* = 0.7712331172401684
c = 3 c* = 1.8392601978916718


In [317]:
N_bi = sum(bi_count)
N_tri = sum(tri_count)
N_quad = sum(quad_count)

print("Total bigram tokens:", N_bi)
print("Total trigram tokens:", N_tri)
print("Total quadgram tokens:", N_quad)

Total bigram tokens: 17419763
Total trigram tokens: 17419763
Total quadgram tokens: 17419763


In [318]:
def create_gt_probability(n,vocab,rows,frequencies):

  vocab_size=len(vocab)
  adjusted=adjusted_counts(frequencies)


  tokens=sum(count*number for count,number in frequencies.items())


  types=sum(frequencies.values())

  possible=((vocab_size+1)**(n-1)*vocab_size)
  unseen_types=possible-types

  singleton_count=frequencies[1]

  missing_mass=(singleton_count/tokens)

  if singleton_count==0:

    missing_mass=1/(tokens+1)

  elif singleton_count==tokens:
    missing_mass=tokens/(tokens+1)



  if unseen_types==0:
    missing_mass=0.0

  normalizer=sum(number*adjusted[count] for count,number in frequencies.items())

  seen={}

  for count,value in adjusted.items():

    seen[count]=((1-missing_mass)*value/normalizer)

  if unseen_types:
    unseen=(missing_mass/unseen_types)

  else:

    unseen=0.0
  normalizers={}


  def probability(word,history):

    row=rows.get(history)


    if not row:

      return 1.0/vocab_size

    if history not in normalizers:

      normalizers[history]=math.fsum(
          seen[count]
          for count in row.values()) + (vocab_size-len(row))*unseen


    count=row.get(word,0)


    if count:

      return (seen[count]/normalizers[history])


    return (unseen/normalizers[history])


  return probability

In [319]:
bi_gt_probability=create_gt_probability(2,unigram,bigram,bi_Nc)

In [320]:
tri_gt_probability=create_gt_probability(3,unigram,trigram,tri_Nc)

In [321]:
quad_gt_probability=create_gt_probability(4,unigram,quadgram,quad_Nc)

In [322]:
def good_turing_perplexity(data,order,unigram, bigram,trigram,quadgram,bi_gt_probability, tri_gt_probability,quad_gt_probability):

  log_probability=0.0
  total=0
  zero_events=0


  for words in data:

    tokens=list(words)


    tokens=(order-1)*["<s>"]+tokens+["</s>"]


    for i in range(len(tokens)-order+1):

      gram=tuple(tokens[i:i+order])


      if order==2:
        probability=bi_gt_probability(gram[-1], gram[:-1])


      elif order==3:

        probability=tri_gt_probability(gram[-1], gram[:-1])

      elif order==4:

        probability=quad_gt_probability(gram[-1],gram[:-1])

      else:
        raise ValueError(
            "Order must be 2, 3 or 4"
        )


      total+=1
      if probability==0:
        zero_events+=1
      else:
        log_probability+=math.log(
            probability
        )

  if total==0:

    return float("inf")


  if zero_events:

    return float("inf")


  return math.exp(
      -log_probability/total
  )

In [323]:
bi_gt_ppl=good_turing_perplexity(dev_df["tokens"],2,unigram,bigram,trigram,quadgram,bi_gt_probability,tri_gt_probability,quad_gt_probability)

tri_gt_ppl=good_turing_perplexity(dev_df["tokens"],3,unigram,bigram,trigram,quadgram,bi_gt_probability,tri_gt_probability,quad_gt_probability)

quad_gt_ppl=good_turing_perplexity(dev_df["tokens"],4,unigram,bigram,trigram,quadgram,bi_gt_probability,tri_gt_probability,quad_gt_probability)

print("Bigram Good-Turing Dev Perplexity:",bi_gt_ppl)

print("Trigram Good-Turing Dev Perplexity:",tri_gt_ppl)

print("Quadgram Good-Turing Dev Perplexity:",quad_gt_ppl)

Bigram Good-Turing Dev Perplexity: 514767.00000178075
Trigram Good-Turing Dev Perplexity: 212287.76916075908
Quadgram Good-Turing Dev Perplexity: 1618745.5037877911


In [267]:
def create_gt_unigram_prob(unigram,uni_Nc):

  adjusted=adjusted_counts(uni_Nc)

  tokens=sum(unigram.values())
  singleton_count=uni_Nc[1]

  missing_mass=singleton_count/tokens

  if singleton_count==0:
    missing_mass=1/(tokens+1)

  elif singleton_count==tokens:
    missing_mass=tokens/(tokens+1)

  normalizer=0

  for count,number in uni_Nc.items():
    normalizer+=number*adjusted[count]

  prob={}

  for word,count in unigram.items():
    prob[word]=(1-missing_mass)*adjusted[count]/normalizer

  return prob,missing_mass

In [ ]:
uni_prob,missing_mass=create_gt_unigram_prob(unigram,uni_Nc)

print("Missing mass:",missing_mass)
print("Observed probability sum:",sum(uni_prob.values()))
print("Total with unseen mass:",sum(uni_prob.values())+missing_mass)

Missing mass: 0.016571178379407345
Observed probability sum: 0.9834288216205922
Total with unseen mass: 0.9999999999999996


**Katz Backoff**

In [270]:
def katz_discount(c,Nc):
  if c==0:
    return 0
  if Nc[c+1]==0:
    return c
  return (c+1)*Nc[c+1]/Nc[c]

In [271]:
def create_katz_discounts(Nc):
  adjusted=adjusted_counts(Nc)

  discounts={}

  for count,value in adjusted.items():
    discounts[count]=min(value,count*(1-1e-12))

  return discounts


bi_discount=create_katz_discounts(bi_Nc)
tri_discount=create_katz_discounts(tri_Nc)
quad_discount=create_katz_discounts(quad_Nc)

print("Discounts completed")

Discounts completed


In [ ]:

uni_prob,missing_mass=create_gt_unigram_prob(unigram,uni_Nc)


vocab=set(unigram.keys())

for tokens in dev_df["tokens"]:
  vocab.update(tokens)

for tokens in test_df["tokens"]:
  vocab.update(tokens)

vocab.add("</s>")

unseen_words=len(vocab-set(unigram))

uni_unseen_prob=missing_mass/unseen_words

print("Missing mass:",missing_mass)
print("Observed probability sum:",sum(uni_prob.values()))
print("Total with unseen mass:",sum(uni_prob.values())+ unseen_words*uni_unseen_prob)

Missing mass: 0.016571178379407345
Observed probability sum: 0.9834288216205922
Total with unseen mass: 0.9999999999999996


In [325]:
def katz_probability(word,history,unigram,bigram,trigram,quadgram,bi_discount,tri_discount,quad_discount,bi_alpha,tri_alpha,quad_alpha):

  if len(history)==0:
    return get_unigram_prob(word)

  if len(history)==1:

    context=history[0]

    row=bigram.get(context)

    if not row:
      return katz_probability(word,(),unigram,bigram,trigram,quadgram, bi_discount,tri_discount,quad_discount,  bi_alpha,tri_alpha,quad_alpha)

    count=row.get(word,0)

    if count>0:
      total=sum(row.values())
      return bi_discount[count]/total

    alpha=bi_alpha.get(context,0)

    return alpha*katz_probability( word,(),unigram,bigram,trigram,quadgram,bi_discount,tri_discount,quad_discount,bi_alpha,tri_alpha,quad_alpha)

  if len(history)==2:

    context=(history[0],history[1])

    row=trigram.get(context)

    if not row:
      return katz_probability(word,history[1:],unigram,bigram,trigram,quadgram,bi_discount,tri_discount,quad_discount,bi_alpha,tri_alpha,quad_alpha )

    count=row.get(word,0)

    if count>0:
      total=sum(row.values())
      return tri_discount[count]/total

    alpha=tri_alpha.get(context,0)

    return alpha*katz_probability(word,history[1:],unigram,bigram,trigram,quadgram,bi_discount,tri_discount,quad_discount,bi_alpha,tri_alpha,quad_alpha)

  context=(history[0],history[1],history[2])

  row=quadgram.get(context)

  if not row:
    return katz_probability(word,history[1:],unigram,bigram,trigram,quadgram,bi_discount,tri_discount,quad_discount,bi_alpha,tri_alpha,quad_alpha)

  count=row.get(word,0)

  if count>0:
    total=sum(row.values())
    return quad_discount[count]/total

  alpha=quad_alpha.get(context,0)

  return alpha*katz_probability(word,history[1:],unigram,bigram,trigram,quadgram,bi_discount,tri_discount,quad_discount,bi_alpha,tri_alpha,quad_alpha)
  

In [274]:
def get_unigram_prob(word):

  if word in uni_prob:
    return uni_prob[word]

  if word in vocab:
    return uni_unseen_prob

  return 0

In [ ]:
def create_katz_bigram_alpha(bigram,unigram,bi_discount,uni_prob):

  alpha={}

  for context,row in bigram.items():

    total=sum(row.values())

    if total==0:
      continue

    # Calculate leftover probability
    leftover=0

    for word,count in row.items():
      leftover+=count-bi_discount[count]

    leftover/=total

    # lower-order probability of seen words
    seen_lower_prob=0

    for word in row:
      seen_lower_prob+=get_unigram_prob(word)

    # lower-order probability available for unseen words
    lower_unseen=1-seen_lower_prob

    if len(row)==len(unigram) or lower_unseen<=0:
      alpha[context]=0
    else:
      alpha[context]=leftover/lower_unseen

  return alpha

In [ ]:
bi_alpha=create_katz_bigram_alpha(bigram,unigram,bi_discount,uni_prob)

print("Bigram alpha completed")

Bigram alpha completed


In [277]:
def katz_bigram_prob_fast(w1,w2):

  row=bigram.get(w1)

  if not row:
    return get_unigram_prob(w2)

  count=row.get(w2,0)

  if count>0:
    total=sum(row.values())
    return bi_discount[count]/total

  alpha=bi_alpha.get(w1,0)

  return alpha*get_unigram_prob(w2)

In [ ]:
def create_katz_trigram_alpha(trigram,tri_discount):

  alpha={}

  for context,row in trigram.items():

    total=sum(row.values())

    if total==0:
      continue

    leftover=0

    for word,count in row.items():
      leftover+=count-tri_discount[count]

    leftover/=total

    w1,w2=context

    seen_lower_prob=0

    for word in row:
      seen_lower_prob+=katz_bigram_prob_fast(w2,word)

    lower_unseen=1-seen_lower_prob

    if len(row)==len(unigram) or lower_unseen<=0:
      alpha[context]=0
    else:
      alpha[context]=leftover/lower_unseen

  return alpha

In [ ]:
tri_alpha=create_katz_trigram_alpha(trigram,tri_discount)

In [280]:
def katz_trigram_prob_fast(w1,w2,w3):

  context=(w1,w2)

  row=trigram.get(context)

  if not row:
    return katz_bigram_prob_fast(w2,w3)

  count=row.get(w3,0)

  if count>0:
    total=sum(row.values())
    return tri_discount[count]/total

  alpha=tri_alpha.get(context,0)

  return alpha*katz_bigram_prob_fast(w2,w3)

In [ ]:
def create_katz_quadgram_alpha(quadgram,quad_discount):
  alpha={}

  for context,row in quadgram.items():

    total=sum(row.values())

    if total==0:
      continue

    leftover=0

    for word,count in row.items():
      leftover+=count-quad_discount[count]

    leftover/=total

    w1,w2,w3=context

    seen_lower_prob=0

    for word in row:
      seen_lower_prob+=katz_trigram_prob_fast(
          w2,w3,word
      )

    lower_unseen=1-seen_lower_prob

    if len(row)==len(unigram) or lower_unseen<=0:
      alpha[context]=0
    else:
      alpha[context]=leftover/lower_unseen

  return alpha

In [282]:
quad_alpha=create_katz_quadgram_alpha(
    quadgram,
    quad_discount
)

print("Quadgram alpha completed")

Quadgram alpha completed


In [ ]:
def katz_quadgram_prob_fast(w1,w2,w3,w4):

  context=(w1,w2,w3)

  row=quadgram.get(context)

  if not row:
    return katz_trigram_prob_fast(w2,w3,w4)

  count=row.get(w4,0)

  if count>0:
    total=sum(row.values())
    return quad_discount[count]/total

  alpha=quad_alpha.get(context,0)

  return alpha*katz_trigram_prob_fast(w2,w3,w4)

In [284]:
print("Bigram alpha:",len(bi_alpha))
print("Trigram alpha:",len(tri_alpha))
print("Quadgram alpha:",len(quad_alpha))

Bigram alpha: 514767
Trigram alpha: 5453509
Quadgram alpha: 10690894


In [285]:
print("Bigram alpha sample:",list(bi_alpha.items())[:3])
print("Trigram alpha sample:",list(tri_alpha.items())[:3])
print("Quadgram alpha sample:",list(quad_alpha.items())[:3])

Bigram alpha sample: [('<s>', 0.4931893724154246), ('૮૦', 0.28777521395544303), ('થી', 0.39843211786022453)]
Trigram alpha sample: [(('<s>', '<s>'), 1.1792950862491234), (('<s>', '૮૦'), 1.0658019770957763), (('૮૦', 'થી'), 0.35409983439959875)]
Quadgram alpha sample: [(('<s>', '<s>', '<s>'), 1.0532742186555113), (('<s>', '<s>', '૮૦'), 1.0568113586445824), (('<s>', '૮૦', 'થી'), 1.7160805737100684)]


In [286]:
def check_katz_zeros(data,order):

  zero=0
  total=0

  for tokens in data:

    tokens=list(tokens)

    tokens=(order-1)*["<s>"]+tokens+["</s>"]

    for i in range(len(tokens)-order+1):

      word=tokens[i+order-1]
      history=tuple(tokens[i:i+order-1])

      if order==2:
        prob=katz_bigram_prob_fast(
            history[0],word
        )

      elif order==3:
        prob=katz_trigram_prob_fast(
            history[0],
            history[1],
            word
        )

      else:
        prob=katz_quadgram_prob_fast(
            history[0],
            history[1],
            history[2],
            word
        )

      total+=1

      if prob<=0:
        zero+=1

  print("Total:",total)
  print("Zero:",zero)

In [287]:
check_katz_zeros(dev_df["tokens"],2)
check_katz_zeros(dev_df["tokens"],3)
check_katz_zeros(dev_df["tokens"],4)

Total: 15814
Zero: 0
Total: 15814
Zero: 0
Total: 15814
Zero: 0


In [288]:
oov_words=set()

for tokens in dev_df["tokens"]:

  for word in tokens:

    if word not in unigram:
      oov_words.add(word)

print("OOV words:",len(oov_words))
print(list(oov_words)[:20])

OOV words: 283
['દંતમાંસની', 'રામકુમારદાસજી', 'વડોદારના', 'આઈફૉનમાં', 'મરડાની', 'ભવ્યઉજવણી', 'મુનસરા', 'લાવઆણ', 'બ્લુમ', 'ડીમાંડનો', 'તલાટીમંત્રીના', 'પ્રક્લિનિકલ', 'સવ્યસાચી', 'કપડાંનોભવ્ય', 'માઇક્રૉસૉફ્ટ', 'મુખ્યચાર', 'કુરણ', 'આલ્વેઇઝે', 'ધ૨ણાં', 'સુકાવું']


In [289]:
oov_count=0
total=0

for tokens in dev_df["tokens"]:

  for word in list(tokens)+["</s>"]:

    total+=1

    if word not in unigram:
      oov_count+=1

print("Total words:",total)
print("OOV events:",oov_count)
print("OOV percentage:",oov_count/total*100)

Total words: 15814
OOV events: 287
OOV percentage: 1.8148476033894017


In [ ]:
def katz_perplexity(sentence,order,unigram,bigram,trigram,quadgram,bi_discount,tri_discount,quad_discount,bi_alpha,tri_alpha,quad):

  log_prob=0
  total_word=0

  for tokens in sentence:

    tokens=list(tokens)

    tokens=(order-1)*["<s>"]+tokens+["</s>"]

    for i in range(len(tokens)-(order-1)):

      history=tuple(tokens[i:i+order-1])
      word=tokens[i+order-1]

      prob=katz_probability( word, history,unigram,bigram,trigram,quadgram,bi_discount,tri_discount,quad_discount,bi_alpha,tri_alpha,quad_alpha)

      if prob<=0:
        return float("inf")

      log_prob+=math.log(prob)
      total_word+=1

  return math.exp(-log_prob/total_word)

In [ ]:
print("Katz Bigram Dev PPL:",katz_perplexity(dev_df["tokens"],2,unigram,bigram,trigram,quadgram,bi_discount,tri_discount,quad_discount,bi_alpha,tri_alpha,quad_alpha))

print("Katz Trigram Dev PPL:",
      katz_perplexity(dev_df["tokens"],3,unigram,bigram,trigram,quadgram,bi_discount,tri_discount,quad_discount,bi_alpha,tri_alpha,quad_alpha))

print("Katz Quadgram Dev PPL:", katz_perplexity(dev_df["tokens"],4,unigram,bigram,trigram,quadgram,bi_discount,tri_discount,quad_discount,bi_alpha,tri_alpha,quad_alpha))

Katz Bigram Dev PPL: 496.5600801924075
Katz Trigram Dev PPL: 428.3534594998422
Katz Quadgram Dev PPL: 442.26264850404937


## Stupid Backoff

In [292]:
SB_ALPHA=0.4

In [ ]:
def stupid_backoff_probability(word,history,unigram,bigram,trigram,quadgram):

  # Unigram
  if len(history)==0:

    total=sum(unigram.values())

    if word in unigram and total>0:
      return unigram[word]/total

    return 0

  # Bigram
  if len(history)==1:

    context=history[0]

    row=bigram.get(context)

    if row:

      count=row.get(word,0)

      if count>0:
        total=sum(row.values())
        return count/total

    return SB_ALPHA*stupid_backoff_probability(word,(),unigram,bigram,trigram,quadgram)

  # Trigram
  if len(history)==2:

    context=(history[0],history[1])

    row=trigram.get(context)

    if row:

      count=row.get(word,0)

      if count>0:
        total=sum(row.values())
        return count/total

    return SB_ALPHA*stupid_backoff_probability(word,history[1:],unigram,bigram,trigram,quadgram)

  # Quadrigram
  context=(history[0],history[1],history[2])

  row=quadgram.get(context)

  if row:

    count=row.get(word,0)

    if count>0:
      total=sum(row.values())
      return count/total

  return SB_ALPHA*stupid_backoff_probability(word,history[1:],unigram,bigram,trigram,quadgram)

In [ ]:
def stupid_backoff_score(data,order,unigram,bigram,trigram,qu):

  log_score=0
  total_word=0

  for tokens in data:

    tokens=list(tokens)

    tokens=(order-1)*["<s>"]+tokens+["</s>"]

    for i in range(len(tokens)-order+1):

      word=tokens[i+order-1]

      history=tuple(
          tokens[i:i+order-1]
      )

      prob=stupid_backoff_probability(word,history,unigram,bigram,trigram,quadgram))

      if prob<=0:
        return float("inf")

      log_score+=math.log(prob)
      total_word+=1

  return math.exp(-log_score/total_word)

In [ ]:
print(    "Stupid Backoff Bigram Dev Score:",stupid_backoff_score(dev_df["tokens"],2,unigram,  bigram,trigram,quadgram ))

print( "Stupid Backoff Trigram Dev Score:",stupid_backoff_score(dev_df["tokens"],3,unigram,bigram,trigram,quadgram))

print("Stupid Backoff Quadgram Dev Score:", stupid_backoff_score(dev_df["tokens"],4,unigram,bigram,trigram,quadgram))

Stupid Backoff Bigram Dev Score: inf
Stupid Backoff Trigram Dev Score: inf
Stupid Backoff Quadgram Dev Score: inf


## Kneser Ney

In [299]:
KN_D = 0.75

UNIFORM_WEIGHT = 1e-8
print("Kneser-Ney discount:", KN_D)

Kneser-Ney discount: 0.75


In [ ]:
continuation_count = Counter()

for context, row in bigram.items():
    for word in row:
        continuation_count[word] += 1

total_unique_bigrams = sum(continuation_count.values())

In [ ]:
def kn_unigram_prob(word):
    if total_unique_bigrams == 0:
        return 0

    continuation_prob = continuation_count.get(word, UNIFORM_WEIGHT) / total_unique_bigrams

    return ((1 - UNIFORM_WEIGHT) * continuation_prob+ UNIFORM_WEIGHT / len(vocab))

In [302]:
def create_kn_bigram_lambda(bigram):

    alpha={}

    for context,row in bigram.items():

        total=sum(row.values())

        if total==0:
            continue

        unique_followers=len(row)

        alpha[context]=(KN_D*unique_followers)/total

    return alpha


kn_bi_lambda=create_kn_bigram_lambda(bigram)

print("Bigram lambda:",len(kn_bi_lambda))
print("Sample:",list(kn_bi_lambda.items())[:3])

Bigram lambda: 514767
Sample: [('<s>', 0.07341346298376364), ('૮૦', 0.22739361702127658), ('થી', 0.1608242005704849)]


In [303]:
def kn_bigram_prob(w1,w2):

    row=bigram.get(w1)

    if not row:
        return kn_unigram_prob(w2)

    total=sum(row.values())

    count=row.get(w2,0)

    discounted=max(count-KN_D,0)/total

    backoff=kn_bi_lambda.get(w1,0)*kn_unigram_prob(w2)

    return discounted+backoff

In [304]:
def create_kn_trigram_lambda(trigram):

    alpha={}

    for context,row in trigram.items():

        total=sum(row.values())

        if total==0:
            continue

        unique_followers=len(row)

        alpha[context]=(KN_D*unique_followers)/total

    return alpha


kn_tri_lambda=create_kn_trigram_lambda(trigram)

print("Trigram lambda:",len(kn_tri_lambda))
print("Sample:",list(kn_tri_lambda.items())[:3])

Trigram lambda: 5453509
Sample: [(('<s>', '<s>'), 0.07341346298376364), (('<s>', '૮૦'), 0.3698630136986301), (('૮૦', 'થી'), 0.23076923076923078)]


In [305]:
def kn_trigram_prob(w1,w2,w3):

    context=(w1,w2)

    row=trigram.get(context)

    if not row:
        return kn_bigram_prob(w2,w3)

    total=sum(row.values())

    count=row.get(w3,0)

    discounted=max(count-KN_D,0)/total

    backoff=kn_tri_lambda.get(context,0)*kn_bigram_prob(w2,w3)

    return discounted+backoff

In [306]:
def create_kn_quadgram_lambda(quadgram):

    alpha={}

    for context,row in quadgram.items():

        total=sum(row.values())

        if total==0:
            continue

        unique_followers=len(row)

        alpha[context]=(KN_D*unique_followers)/total

    return alpha


kn_quad_lambda=create_kn_quadgram_lambda(quadgram)

print("Quadgram lambda:",len(kn_quad_lambda))
print("Sample:",list(kn_quad_lambda.items())[:3])

Quadgram lambda: 10690894
Sample: [(('<s>', '<s>', '<s>'), 0.07341346298376364), (('<s>', '<s>', '૮૦'), 0.3698630136986301), (('<s>', '૮૦', 'થી'), 0.75)]


In [307]:
def kn_quadgram_prob(w1,w2,w3,w4):

    context=(w1,w2,w3)

    row=quadgram.get(context)

    if not row:
        return kn_trigram_prob(w2,w3,w4)

    total=sum(row.values())

    count=row.get(w4,0)

    discounted=max(count-KN_D,0)/total

    backoff=kn_quad_lambda.get(context,0)*kn_trigram_prob(
        w2,w3,w4
    )

    return discounted+backoff

In [ ]:
def kn_probability(word,history):

    if len(history)==0:
        return kn_unigram_prob(word)

    if len(history)==1:
        return kn_bigram_prob(history[0],word)

    if len(history)==2:
        return kn_trigram_prob(history[0], history[1],word)

    return kn_quadgram_prob(history[0],history[1],history[2],word)

In [ ]:
def check_kn_zeros(data,order):

    zero=0
    total=0

    for tokens in data:

        tokens=list(tokens)

        tokens=(order-1)*["<s>"]+tokens+["</s>"]

        for i in range(len(tokens)-order+1):

            word=tokens[i+order-1]

            history=tuple(tokens[i:i+order-1])

            prob=kn_probability(word,history)

            total+=1

            if prob<=0:
                zero+=1

    print("Total:",total)
    print("Zero:",zero)

In [310]:
check_kn_zeros(dev_df["tokens"],2)
check_kn_zeros(dev_df["tokens"],3)
check_kn_zeros(dev_df["tokens"],4)

Total: 15814
Zero: 0
Total: 15814
Zero: 0
Total: 15814
Zero: 0


In [ ]:
def kn_perplexity(data,order):

    log_prob=0
    total_word=0

    for tokens in data:

        tokens=list(tokens)

        tokens=(order-1)*["<s>"]+tokens+["</s>"]

        for i in range(len(tokens)-order+1):

            history=tuple(
                tokens[i:i+order-1]
            )

            word=tokens[i+order-1]

            prob=kn_probability(
                word,
                history
            )

            if prob<=0:
                return float("inf")

            log_prob+=math.log(prob)

            total_word+=1

    return math.exp(-log_prob/total_word)

In [ ]:
print("Kneser-Ney Bigram Dev PPL:",kn_perplexity(dev_df["tokens"],2))

print("Kneser-Ney Trigram Dev PPL:", kn_perplexity(dev_df["tokens"],3))

print("Kneser-Ney Quadgram Dev PPL:",kn_perplexity(dev_df["tokens"],4))

Kneser-Ney Bigram Dev PPL: 692.9240206363503
Kneser-Ney Trigram Dev PPL: 613.1514337602351
Kneser-Ney Quadgram Dev PPL: 641.1782150964407


In [ ]:
print("Kneser-Ney Bigram Test PPL:",kn_perplexity(test_df["tokens"],2))

print("Kneser-Ney Trigram Test PPL:",kn_perplexity(test_df["tokens"],3))

print( "Kneser-Ney Quadgram Test PPL:",kn_perplexity(test_df["tokens"],4))

Kneser-Ney Bigram Test PPL: 743.0847827447086
Kneser-Ney Trigram Test PPL: 653.6668938985772
Kneser-Ney Quadgram Test PPL: 679.6855312076696
